In [ ]:
# Main execution function
def run_dbin_test(hsi_dir, model_dir, rgb_dir, sf, num_images, image_size=512, batch_size=1):
    """
    Execute DBIN test
    
    Args:
        hsi_dir: Directory with HSI MAT files
        model_dir: Checkpoint directory
        rgb_dir: Optional RGB directory
        sf: Scale factor (8 or 16)
        num_images: Number of test images
        image_size: Image size (512)
        batch_size: Batch size (1)
    model-sf16-15k/tensorflow2/default/1
    Returns:
        Dictionary with metrics and results
    """
    print(f"\n{'='*70}")
    print(f"DBIN Testing with SF={sf}")
    print(f"{'='*70}")
    print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
    print(f"HSI Dir: {hsi_dir}")
    print(f"Model: {model_dir}")
    print(f"Images: {num_images}")
    
    # Convert MATs to TFRecord
    out_dir = './temp_tfrecords'
    _ensure_dir(out_dir)
    tfrecord_path = os.path.join(out_dir, f'test_sf{sf}.tfrecords')
    
    print(f"\nConverting MATs -> TFRecord...")
    convert_mats_to_test_tfrecord(hsi_dir, tfrecord_path, rgb_dir=rgb_dir, sf=sf, crop_size=image_size)
    
    # Build graph
    print(f"Building computation graph...")
    tf.compat.v1.reset_default_graph()
    
    gt_holder = tf.compat.v1.placeholder(dtype=tf.float32, shape=[batch_size, image_size, image_size, 31])
    ms_holder = tf.compat.v1.placeholder(dtype=tf.float32, shape=[batch_size, image_size // sf, image_size // sf, 31])
    pan_holder = tf.compat.v1.placeholder(dtype=tf.float32, shape=[batch_size, image_size, image_size, 3])
    pan2_holder = tf.compat.v1.placeholder(dtype=tf.float32, shape=[batch_size, image_size // 2, image_size // 2, 3])
    pan4_holder = tf.compat.v1.placeholder(dtype=tf.float32, shape=[batch_size, image_size // 4, image_size // 4, 3])
    
    # Build model
    X = fusion_net(pan_holder, ms_holder, num_spectral=31, num_fm=64, num_ite=8, sf=sf, reuse=False, weight_decay=1e-5)
    output = tf.clip_by_value(X, 0.0, 1.0)
    
    mse = tf.square(output - gt_holder)
    mse = tf.reshape(mse, [image_size, image_size, 31])
    final_mse = tf.reduce_mean(mse, axis=[0, 1])
    
    # Session config
    config = tf.compat.v1.ConfigProto()
    config.gpu_options.allow_growth = True
    
    init = tf.group(
        tf.compat.v1.global_variables_initializer(),
        tf.compat.v1.local_variables_initializer(),
    )
    saver = tf.compat.v1.train.Saver()
    
    # Metrics storage
    average_psnr = 0.0
    average_ssim = 0.0
    average_sam = 0.0
    average_ergas = 0.0
    
    gt_out = np.zeros(shape=[num_images, image_size, image_size, 31], dtype=np.float32)
    net_out = np.zeros(shape=[num_images, image_size, image_size, 31], dtype=np.float32)
    
    # Run session
    with tf.compat.v1.Session(config=config) as sess:
        sess.run(init)
        
        coord = tf.compat.v1.train.Coordinator()
        threads = tf.compat.v1.train.start_queue_runners(sess=sess, coord=coord)
        
        # Restore checkpoint
        try:
            ckpt = _resolve_checkpoint(model_dir)
            print(f"Loading checkpoint: {ckpt}")
            saver.restore(sess, ckpt)
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            coord.request_stop()
            coord.join(threads)
            return {'status': 'error', 'error': str(e)}
        
        # Process images
        print(f"\nProcessing {num_images} images...\n")
        
        pan_batch_tensor, pan2_batch_tensor, pan4_batch_tensor, gt_batch_tensor, ms_batch_tensor, ms2_batch_tensor = read_and_decode_test_sf(
            tfrecord_path, batch_size=batch_size, image_size=image_size, sf=sf
        )
        
        for i in range(num_images):
            try:
                pan, pan2, pan4, gt, ms, ms2 = sess.run(
                    [pan_batch_tensor, pan2_batch_tensor, pan4_batch_tensor, gt_batch_tensor, ms_batch_tensor, ms2_batch_tensor]
                )
                
                out, mse_loss = sess.run(
                    [output, final_mse],
                    feed_dict={
                        pan_holder: pan,
                        pan2_holder: pan2,
                        pan4_holder: pan4,
                        gt_holder: gt,
                        ms_holder: ms,
                    },
                )
                
                gt_out[i, :, :, :] = gt
                net_out[i, :, :, :] = np.array(out)
                
                mse_loss = np.array(mse_loss)
                ms_psnr = np.mean(10 * np.log10(1.0 / mse_loss))
                temp_ssim = compute_ms_ssim(out, gt)
                temp_sam = compute_sam(out, gt)
                temp_ergas = compute_ergas(mse_loss, out, sf=sf)
                
                print(f'Image {i:2d}: PSNR={ms_psnr:.4f} | SSIM={temp_ssim:.4f} | SAM={temp_sam:.4f} | ERGAS={temp_ergas:.4f}')
                
                average_psnr += ms_psnr / float(num_images)
                average_ssim += temp_ssim / float(num_images)
                average_sam += temp_sam / float(num_images)
                average_ergas += temp_ergas / float(num_images)
                
            except tf.errors.OutOfRangeError:
                print(f"Reached end of data at image {i}")
                break
        
        coord.request_stop()
        coord.join(threads)
    
    # Save results
    os.makedirs('./results', exist_ok=True)
    result_file = f'./results/dbin_sf{sf}_results.mat'
    sio.savemat(result_file, {'gt_out': gt_out[:i+1], 'net_out': net_out[:i+1]})
    
    return {
        'status': 'success',
        'sf': sf,
        'average_psnr': average_psnr,
        'average_ssim': average_ssim,
        'average_sam': average_sam,
        'average_ergas': average_ergas,
        'num_images': i + 1,
        'result_file': result_file,
        'timestamp': datetime.now().strftime('%H:%M:%S')
    }

print("✓ Main test function defined")

In [ ]:
# Checkpoint resolution
def _resolve_checkpoint(model_dir):
    """Find checkpoint prefix from directory."""
    import re
    
    ckpt = tf.compat.v1.train.latest_checkpoint(model_dir)
    if ckpt:
        idx = ckpt + ".index"
        dat = ckpt + ".data-00000-of-00001"
        if os.path.exists(idx) and os.path.exists(dat):
            return ckpt
    
    candidates = []
    for fname in os.listdir(model_dir):
        if fname.endswith('.ckpt.index'):
            prefix = fname[:-len('.index')]
            dat = os.path.join(model_dir, prefix + '.data-00000-of-00001')
            if not os.path.exists(dat):
                continue
            m = re.search(r'model-(\d+)\.ckpt$', prefix)
            step = int(m.group(1)) if m else -1
            candidates.append((step, os.path.join(model_dir, prefix)))
    
    if candidates:
        candidates.sort()
        return candidates[-1][1]
    
    raise RuntimeError(f'No checkpoint found in {model_dir}')

print("✓ Checkpoint resolver defined")

In [ ]:
# DBIN model definition
def Fusion(Z, Y, weight_decay, num_spectral=31, num_fm=64, reuse=True):
    """Fusion module with channel attention."""
    with tf.compat.v1.variable_scope('py'):
        if reuse:
            tf.compat.v1.get_variable_scope().reuse_variables()
        
        lms = upsample(Y, Z)
        Xin = tf.concat([lms, Z], axis=3)
        
        Xt = conv_sn(Xin, num_fm, weight_decay, scope='in')
        Xt = lrelu(Xt)
        
        for i in range(4):
            Xi = conv_sn(Xt, num_fm, weight_decay, scope='res{}1'.format(i))
            Xi = lrelu(Xi)
            Xi = conv_sn(Xi, num_fm, weight_decay, scope='res{}2'.format(i))
            
            mask = global_avg_pool(Xi)
            mask = dense_manual(mask, units=num_fm // 16, activation_fn=tf.nn.relu, scope='se{}1'.format(i))
            mask = dense_manual(mask, units=num_fm, activation_fn=None, scope='se{}2'.format(i))
            mask = tf.reshape(mask, [-1, 1, 1, num_fm])
            mask = tf.sigmoid(mask)
            Xi = tf.multiply(Xi, mask)
            
            Xt = Xt + Xi
        
        X = conv_sn(Xt, num_spectral, weight_decay, scope='out')
        return X

def boost_lap(X, Z_in, Y_in, weight_decay, num_spectral=31, num_fm=64, sf=8, reuse=True):
    """Laplacian boosting module."""
    with tf.compat.v1.variable_scope('recursive'):
        if reuse:
            tf.compat.v1.get_variable_scope().reuse_variables()
        
        Z = conv_sn(X, 3, weight_decay, scope='dz')
        Z = lrelu(Z)
        
        dy_kernel = int(sf) + 4
        Y = conv_sn(X, num_spectral, weight_decay, kernel=dy_kernel, stride=int(sf), scope='dy')
        Y = lrelu(Y)
        
        dZ = Z_in - Z
        dY = Y_in - Y
        
        dX = Fusion(dZ, dY, weight_decay, num_spectral=num_spectral, num_fm=num_fm, reuse=True)
        X = X + dX
        return X

def fusion_net(Z, Y, num_spectral=31, num_fm=64, num_ite=8, sf=8, reuse=False, weight_decay=1e-5):
    """Complete DBIN fusion network."""
    with tf.compat.v1.variable_scope('fusion_net'):
        if reuse:
            tf.compat.v1.get_variable_scope().reuse_variables()
        
        X = Fusion(Z, Y, weight_decay, num_spectral=num_spectral, num_fm=num_fm, reuse=False)
        Xs = X
        
        for _ in range(num_ite):
            X = boost_lap(
                X,
                Z,
                Y,
                weight_decay,
                num_spectral=num_spectral,
                num_fm=num_fm,
                sf=sf,
                reuse=True,
            )
            Xs = tf.concat([Xs, X], axis=3)
        
        X = conv_sn(Xs, num_spectral, weight_decay, use_bias=False, scope='out_conv')
        return X

print("✓ DBIN model defined")

In [ ]:
# Neural network building blocks
def _vsi():
    """Variance scaling initializer."""
    try:
        return tf.compat.v1.variance_scaling_initializer()
    except Exception:
        return tf.compat.v1.glorot_uniform_initializer()

def lrelu(x, alpha=0.2):
    return tf.nn.leaky_relu(x, alpha)

def global_avg_pool(x):
    return tf.reduce_mean(x, axis=[1, 2])

def dense_manual(x, units, activation_fn=None, scope='dense'):
    """Keras-3-safe dense layer with TF1-style naming."""
    with tf.compat.v1.variable_scope(scope, reuse=tf.compat.v1.AUTO_REUSE):
        in_dim = x.get_shape().as_list()[-1]
        if in_dim is None:
            raise ValueError('dense_manual requires static last dimension')
        w = tf.compat.v1.get_variable(
            'kernel',
            shape=[in_dim, units],
            initializer=_vsi(),
        )
        b = tf.compat.v1.get_variable(
            'bias',
            shape=[units],
            initializer=tf.zeros_initializer(),
        )
        y = tf.matmul(x, w) + b
        if activation_fn is not None:
            y = activation_fn(y)
        return y

def spectral_norm(w, iteration=1):
    """Spectral normalization."""
    w_shape = w.shape.as_list()
    w_reshaped = tf.reshape(w, [-1, w_shape[-1]])
    
    u = tf.compat.v1.get_variable(
        'u',
        [1, w_shape[-1]],
        initializer=tf.random_normal_initializer(),
        trainable=False,
    )
    
    u_hat = u
    for _ in range(iteration):
        v_ = tf.matmul(u_hat, tf.transpose(w_reshaped))
        v_hat = tf.nn.l2_normalize(v_)
        u_ = tf.matmul(v_hat, w_reshaped)
        u_hat = tf.nn.l2_normalize(u_)
    
    u_hat = tf.stop_gradient(u_hat)
    v_hat = tf.stop_gradient(v_hat)
    
    sigma = tf.matmul(tf.matmul(v_hat, w_reshaped), tf.transpose(u_hat))
    with tf.control_dependencies([u.assign(u_hat)]):
        w_norm = w_reshaped / sigma * 0.7
        w_norm = tf.reshape(w_norm, w_shape)
    return w_norm

def conv_sn(x, channels, weight_decay, kernel=3, stride=1, use_bias=True, scope='conv'):
    """Spectral normalized convolution."""
    with tf.compat.v1.variable_scope(scope, reuse=tf.compat.v1.AUTO_REUSE):
        w = tf.compat.v1.get_variable(
            'kernel',
            shape=[kernel, kernel, x.get_shape()[-1], channels],
            initializer=_vsi(),
        )
        bias = tf.compat.v1.get_variable('bias', [channels], initializer=tf.constant_initializer(0.0))
        y = tf.nn.conv2d(input=x, filters=spectral_norm(w), strides=[1, stride, stride, 1], padding='SAME')
        if use_bias:
            y = tf.nn.bias_add(y, bias)
        return y

def upsample(x, ref):
    """Bilinear upsample to match reference tensor size."""
    target_hw = tf.shape(ref)[1:3]
    return tf.image.resize(x, target_hw, method=tf.image.ResizeMethod.BILINEAR)

print("✓ Neural network blocks defined")

In [ ]:
# TFRecord reader for test data
def read_and_decode_test_sf(tfrecords_file, batch_size, image_size, sf):
    """Read test TFRecord with SF-dependent shapes."""
    if sf < 2 or (sf % 2) != 0:
        raise ValueError('sf must be even >= 2')
    
    ms_size = image_size // sf
    ms2_size = image_size // (sf // 2)
    
    filename_queue = tf.compat.v1.train.string_input_producer([tfrecords_file])
    reader = tf.compat.v1.TFRecordReader()
    _, serialized_example = reader.read(filename_queue)
    
    img_features = tf.compat.v1.parse_single_example(
        serialized_example,
        features={
            'pan_raw': tf.io.FixedLenFeature([], tf.string),
            'pan2_raw': tf.io.FixedLenFeature([], tf.string),
            'pan4_raw': tf.io.FixedLenFeature([], tf.string),
            'gt_raw': tf.io.FixedLenFeature([], tf.string),
            'ms_raw': tf.io.FixedLenFeature([], tf.string),
            'ms2_raw': tf.io.FixedLenFeature([], tf.string),
        },
    )
    
    pan = tf.io.decode_raw(img_features['pan_raw'], tf.float32)
    pan = tf.reshape(pan, [image_size, image_size, 3])
    pan2 = tf.io.decode_raw(img_features['pan2_raw'], tf.float32)
    pan2 = tf.reshape(pan2, [image_size // 2, image_size // 2, 3])
    pan4 = tf.io.decode_raw(img_features['pan4_raw'], tf.float32)
    pan4 = tf.reshape(pan4, [image_size // 4, image_size // 4, 3])
    gt = tf.io.decode_raw(img_features['gt_raw'], tf.float32)
    gt = tf.reshape(gt, [image_size, image_size, 31])
    ms = tf.io.decode_raw(img_features['ms_raw'], tf.float32)
    ms = tf.reshape(ms, [ms_size, ms_size, 31])
    ms2 = tf.io.decode_raw(img_features['ms2_raw'], tf.float32)
    ms2 = tf.reshape(ms2, [ms2_size, ms2_size, 31])
    
    pan_batch, pan2_batch, pan4_batch, gt_batch, ms_batch, ms2_batch = tf.compat.v1.train.batch(
        [pan, pan2, pan4, gt, ms, ms2],
        batch_size=batch_size,
        num_threads=4,
        capacity=300,
        allow_smaller_final_batch=False,
    )
    return pan_batch, pan2_batch, pan4_batch, gt_batch, ms_batch, ms2_batch

print("✓ TFRecord reader defined")

In [ ]:
# TFRecord conversion function
def convert_mats_to_test_tfrecord(mat_dir, save_path, rgb_dir=None, sf=8, crop_size=512):
    """Convert MAT files to TFRecord format for DBIN testing."""
    _ensure_dir(os.path.dirname(save_path))
    
    if sf < 2 or (sf % 2) != 0:
        raise ValueError('sf must be even >= 2')
    
    def _center_crop_square(arr, size):
        if size is None or size <= 0:
            return arr
        if arr.ndim == 4:
            out = []
            for i in range(arr.shape[0]):
                out.append(_center_crop_square(arr[i], size))
            return np.stack(out, axis=0)
        h, w = arr.shape[0], arr.shape[1]
        if h < size or w < size:
            raise ValueError('Cannot crop {} from {}'.format(size, arr.shape))
        y0 = (h - size) // 2
        x0 = (w - size) // 2
        return arr[y0:y0 + size, x0:x0 + size, ...]
    
    def resize_hw(arr, new_hw):
        h, w = new_hw
        out = []
        for c in range(arr.shape[2]):
            out.append(cv2.resize(arr[:, :, c], (w, h), interpolation=cv2.INTER_AREA))
        return np.stack(out, axis=2).astype(np.float32)
    
    try:
        writer = tf.io.TFRecordWriter(save_path)
    except Exception:
        writer = tf.python_io.TFRecordWriter(save_path)
    
    mats = [f for f in os.listdir(mat_dir) if f.endswith('.mat')]
    count = 0
    
    for fname in sorted(mats):
        path = os.path.join(mat_dir, fname)
        m = sio.loadmat(path)
        m_filtered = {k: v for k, v in m.items() if not k.startswith('__')}
        
        for k, v in m_filtered.items():
            arr = np.array(v)
            if arr.ndim not in (3, 4) or arr.shape[-1] < 31:
                continue
            
            gt = _normalize01(arr)
            if gt.ndim == 3:
                gt = gt[np.newaxis, ...]
            
            gt = _center_crop_square(gt, crop_size)
            
            for i in range(gt.shape[0]):
                gt_i = gt[i]
                H, W = gt_i.shape[0], gt_i.shape[1]
                
                if (H % sf) != 0 or (W % sf) != 0:
                    continue
                
                idx_b, idx_g, idx_r = 7, 15, 23
                pan_i = _normalize01(np.stack([gt_i[..., idx_b], gt_i[..., idx_g], gt_i[..., idx_r]], axis=-1))
                
                pan2 = resize_hw(pan_i, (H // 2, W // 2))
                pan4 = resize_hw(pan_i, (H // 4, W // 4))
                gt2 = resize_hw(gt_i, (H // 2, W // 2))
                gt4 = resize_hw(gt_i, (H // 4, W // 4))
                ms_i = resize_hw(gt_i, (H // sf, W // sf))
                ms2 = resize_hw(ms_i, (H // (sf // 2), W // (sf // 2)))
                
                example = tf.train.Example(features=tf.train.Features(feature={
                    'pan_raw': _bytes_feature(pan_i.tobytes()),
                    'pan2_raw': _bytes_feature(pan2.tobytes()),
                    'pan4_raw': _bytes_feature(pan4.tobytes()),
                    'gt_raw': _bytes_feature(gt_i.tobytes()),
                    'ms_raw': _bytes_feature(ms_i.tobytes()),
                    'ms2_raw': _bytes_feature(ms2.tobytes()),
                }))
                writer.write(example.SerializeToString())
                count += 1
    
    writer.close()
    print(f'✓ Wrote {count} samples to {save_path}')
    return save_path

print("✓ TFRecord conversion function defined")

In [ ]:
# Metric computation functions
def compute_ms_ssim(image1, image2):
    """Multi-spectral SSIM: average SSIM across all channels."""
    image1 = np.asarray(image1)
    image2 = np.asarray(image2)
    if image1.ndim == 4:
        image1 = image1[0]
    if image2.ndim == 4:
        image2 = image2[0]
    n = image1.shape[2]
    ms_ssim = 0.0
    for i in range(n):
        single_ssim = compare_ssim(image1[:, :, i], image2[:, :, i], data_range=1.0)
        ms_ssim += single_ssim
    return ms_ssim / n

def compute_sam(image1, image2):
    """Spectral Angle Mapper: mean angle between spectral vectors."""
    image1 = np.asarray(image1)
    image2 = np.asarray(image2)
    if image1.ndim == 4:
        image1 = image1[0]
    if image2.ndim == 4:
        image2 = image2[0]
    h, w, c = image1.shape
    image1 = np.reshape(image1, (h * w, c))
    image2 = np.reshape(image2, (h * w, c))
    mole = np.sum(np.multiply(image1, image2), axis=1)
    image1_norm = np.sqrt(np.sum(np.square(image1), axis=1))
    image2_norm = np.sqrt(np.sum(np.square(image2), axis=1))
    deno = np.multiply(image1_norm, image2_norm)
    sam = np.rad2deg(np.arccos((mole + 1e-11) / (deno + 1e-11)))
    return np.mean(sam)

def compute_ergas(mse, out, sf=8):
    """Erreur Relative Globale Adimensionnelle de Synthèse."""
    out = np.asarray(out)
    if out.ndim == 4:
        out = out[0]
    h, w, c = out.shape
    out = np.reshape(out, (h * w, c))
    out_mean = np.mean(out, axis=0)
    mse = np.reshape(mse, (c, 1))
    out_mean = np.reshape(out_mean, (c, 1))
    ergas = 100.0 / float(sf) * np.sqrt(np.mean(mse / (out_mean ** 2 + 1e-12)))
    return ergas

print("✓ Metric functions defined")

In [ ]:
# Helper functions for data normalization and loading
def _normalize01(x):
    """Normalize input to [0,1] range for DBIN compatibility."""
    x = np.asarray(x, dtype=np.float32)
    mx = float(np.nanmax(x)) if x.size else 0.0
    if mx <= 1.0:
        return np.clip(x, 0.0, 1.0)
    if mx <= 255.0:
        denom = 255.0
    elif mx <= 4095.0:
        denom = 4095.0
    elif mx <= 65535.0:
        denom = 65535.0
    else:
        denom = mx
    x = x / denom
    return np.clip(x, 0.0, 1.0)

def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _ensure_dir(path):
    if not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)

def _load_mat(path, key, want_channels=None, allow_autodetect=False):
    m = sio.loadmat(path)
    if key in m:
        return np.array(m[key], dtype=np.float32)
    if allow_autodetect:
        best = None
        best_size = -1
        for k, v in m.items():
            arr = np.array(v)
            if arr.ndim in (3, 4) and arr.shape[-1] >= (want_channels or 1):
                size = arr.size
                if size > best_size:
                    best = k
                    best_size = size
        if best is not None:
            return np.array(m[best], dtype=np.float32)
    raise KeyError('Key {} not found in {}'.format(key, path))

print("✓ Helper functions defined")

In [ ]:
# Import core libraries
import os
import sys
import numpy as np
import scipy.io as sio
import tensorflow as tf
from datetime import datetime

try:
    import cv2
except Exception:
    cv2 = None
    
try:
    from skimage.measure import compare_ssim
except Exception:
    from skimage.metrics import structural_similarity as compare_ssim

print("✓ Core libraries imported")

# Allow TF1 style under TF2
if tf.__version__.startswith('2'):
    tf.compat.v1.disable_eager_execution()

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')  # CPU-only for Kaggle
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Install dependencies
import subprocess
import sys

!pip install -q tensorflow==1.15.* numpy scipy scikit-image opencv-python sewar
print("✓ Dependencies installed")

- SF8 testing with pretrained weights
- SF16 testing with model checkpoint  
- Metrics: PSNR, SSIM, SAM, ERGAS

In [ ]:
# Test configuration
TEST_CONFIGS = {
    'SF8': {
        'hsi_dir': '/kaggle/input/cave-dataset-2/Data/Test/HSI',
        'rgb_dir': '/kaggle/input/cave-dataset-2/Data/Test/RGB',
        'model_dir': '/kaggle/input/model-sf8-260k/tensorflow2/default/1',  # Update if needed
        'sf': 8,
        'num_images': 12
    },
    'SF16': {
        'hsi_dir': '/kaggle/input/cave-dataset-2/Data/Test/HSI',
        'rgb_dir': '/kaggle/input/cave-dataset-2/Data/Test/RGB',
        'model_dir': '/kaggle/input/model-sf16-15k/tensorflow2/default/1',
        'sf': 16,
        'num_images': 12
    }
}

print("✓ Test configurations loaded")
for k, v in TEST_CONFIGS.items():
    print(f"{k}: sf={v['sf']}, images={v['num_images']}")
    print(f"    model_dir={v['model_dir']}")

In [ ]:
# Run SF8 test
sf8_cfg = TEST_CONFIGS['SF8']
result_sf8 = run_dbin_test(
    hsi_dir=sf8_cfg['hsi_dir'],
    model_dir=sf8_cfg['model_dir'],
    rgb_dir=sf8_cfg['rgb_dir'],
    sf=sf8_cfg['sf'],
    num_images=sf8_cfg['num_images']
)

print("\nSF8 Result Status:", result_sf8['status'])
if result_sf8['status'] == 'success':
    print(f"PSNR: {result_sf8['average_psnr']:.4f}")
    print(f"SSIM: {result_sf8['average_ssim']:.4f}")
    print(f"SAM:  {result_sf8['average_sam']:.4f}")
    print(f"ERGAS:{result_sf8['average_ergas']:.4f}")

In [ ]:
# Run SF16 test
sf16_cfg = TEST_CONFIGS['SF16']
result_sf16 = run_dbin_test(
    hsi_dir=sf16_cfg['hsi_dir'],
    model_dir=sf16_cfg['model_dir'],
    rgb_dir=sf16_cfg['rgb_dir'],
    sf=sf16_cfg['sf'],
    num_images=sf16_cfg['num_images']
)

print("\nSF16 Result Status:", result_sf16['status'])
if result_sf16['status'] == 'success':
    print(f"PSNR: {result_sf16['average_psnr']:.4f}")
    print(f"SSIM: {result_sf16['average_ssim']:.4f}")
    print(f"SAM:  {result_sf16['average_sam']:.4f}")
    print(f"ERGAS:{result_sf16['average_ergas']:.4f}")

In [ ]:
# Final summary (print only)
all_results = [result_sf8, result_sf16]

print("\n" + "=" * 80)
print("DBIN TEST SUMMARY")
print("=" * 80)

for r in all_results:
    sf = r.get('sf', 'N/A')
    status = r.get('status', 'N/A')
    print(f"SF{sf} | Status: {status}")
    if status == 'success':
        print(f"  PSNR : {r.get('average_psnr', float('nan')):.4f}")
        print(f"  SSIM : {r.get('average_ssim', float('nan')):.4f}")
        print(f"  SAM  : {r.get('average_sam', float('nan')):.4f}")
        print(f"  ERGAS: {r.get('average_ergas', float('nan')):.4f}")
        print(f"  Images: {r.get('num_images', 0)}")
        print(f"  Result File: {r.get('result_file', 'N/A')}")
    else:
        print(f"  Error: {r.get('error', 'Unknown error')}")
    print("-" * 80)

print("Done.")